# Week 9 — STRIPS Planning: Where the Guarantee Comes From

**Lesson plan:** [`../weeks/week-09.md`](../weeks/week-09.md) · **Slides:** [`../slides/week-09/deck.md`](../slides/week-09/deck.md)

Live coding, 25 minutes. **This is the intellectual peak of the course.**

The reveal: `astar` is **week 4's function, unmodified**. Only the representation
and the heuristic change — and the heuristic is *derived automatically from the
domain description*.

**Duel 2 (The Formalizer) is due this week.**

## 1 · Actions as schemas

A STRIPS action is three sets of *fluents*: preconditions, adds, deletes.

In [1]:
import heapq, itertools, time
from dataclasses import dataclass


@dataclass(frozen=True)
class Action:
    name: str
    precond: frozenset
    add: frozenset
    delete: frozenset

    def __repr__(self):
        return self.name


def applicable(action, state):
    return action.precond <= state           # set containment. That is all.


def apply_action(action, state):
    return (state - action.delete) | action.add


# --- Blocks World domain -------------------------------------------------
def blocks_actions(blocks):
    """The DOMAIN: written once, reused for every problem instance."""
    acts = []
    for b in blocks:
        acts.append(Action(f"pick-up({b})",
                           frozenset({f"clear({b})", f"ontable({b})", "handempty"}),
                           frozenset({f"holding({b})"}),
                           frozenset({f"clear({b})", f"ontable({b})", "handempty"})))
        acts.append(Action(f"put-down({b})",
                           frozenset({f"holding({b})"}),
                           frozenset({f"clear({b})", f"ontable({b})", "handempty"}),
                           frozenset({f"holding({b})"})))
        for c in blocks:
            if b == c:
                continue
            acts.append(Action(f"stack({b},{c})",
                               frozenset({f"holding({b})", f"clear({c})"}),
                               frozenset({"handempty", f"clear({b})", f"on({b},{c})"}),
                               frozenset({f"holding({b})", f"clear({c})"})))
            acts.append(Action(f"unstack({b},{c})",
                               frozenset({f"on({b},{c})", f"clear({b})", "handempty"}),
                               frozenset({f"holding({b})", f"clear({c})"}),
                               frozenset({f"on({b},{c})", f"clear({b})", "handempty"})))
    return acts


print(f"3 blocks -> {len(blocks_actions('abc'))} ground actions")
for a in blocks_actions("abc")[:3]:
    print(f"\n  {a}")
    print(f"    pre: {sorted(a.precond)}")
    print(f"    add: {sorted(a.add)}")
    print(f"    del: {sorted(a.delete)}")

3 blocks -> 18 ground actions

  pick-up(a)
    pre: ['clear(a)', 'handempty', 'ontable(a)']
    add: ['holding(a)']
    del: ['clear(a)', 'handempty', 'ontable(a)']

  put-down(a)
    pre: ['holding(a)']
    add: ['clear(a)', 'handempty', 'ontable(a)']
    del: ['holding(a)']

  stack(a,b)
    pre: ['clear(b)', 'holding(a)']
    add: ['clear(a)', 'handempty', 'on(a,b)']
    del: ['clear(b)', 'holding(a)']


## 2 · ⚠️ The frame problem

**What does `pick-up(a)` do to `ontable(b)`?**

**Nothing. And nothing was written down.**

That silence *is* the solution. STRIPS assumes anything not in the add or delete
list **persists**.

In pure logic you would need a frame axiom per predicate per action:

```
∀x (x ≠ a → (ontable(x, s) → ontable(x, Result(pickup(a), s))))
```

That is O(A × F) axioms, all saying "nothing happened."

**Does STRIPS *solve* the frame problem, or sidestep it?** It sidesteps it — and
the sidestep is what made planning practical.

In [2]:
state = frozenset({"ontable(a)", "ontable(b)", "clear(a)", "clear(b)", "handempty"})
acts = {a.name: a for a in blocks_actions("ab")}

print("before:", sorted(state))
after = apply_action(acts["pick-up(a)"], state)
print("after :", sorted(after))
print("""
ontable(b) survived, untouched, and we never mentioned it. The STRIPS assumption
did that work -- and it is why apply_action() is ONE LINE instead of a theorem
prover.""")

before: ['clear(a)', 'clear(b)', 'handempty', 'ontable(a)', 'ontable(b)']
after : ['clear(b)', 'holding(a)', 'ontable(b)']

ontable(b) survived, untouched, and we never mentioned it. The STRIPS assumption
did that work -- and it is why apply_action() is ONE LINE instead of a theorem
prover.


## 3 · A Problem in week 3's interface

So we can hand it to week 4's A\* without changing either.

In [3]:
class PlanningProblem:
    def __init__(self, initial, goal, actions):
        self.initial = frozenset(initial)
        self.goal = frozenset(goal)
        self.action_list = actions

    def actions(self, state):
        return [a for a in self.action_list if applicable(a, state)]

    def result(self, state, action):
        return apply_action(action, state)

    def is_goal(self, state):
        return self.goal <= state

    def step_cost(self, state, action):
        return 1


def blocks_problem(n_blocks):
    """Stack n blocks into a single tower a-on-b-on-c-... from a flat table."""
    bl = "abcdefghij"[:n_blocks]
    init = {f"ontable({b})" for b in bl} | {f"clear({b})" for b in bl} | {"handempty"}
    goal = {f"on({bl[i]},{bl[i+1]})" for i in range(len(bl) - 1)}
    return PlanningProblem(init, goal, blocks_actions(bl))


p = blocks_problem(3)
print("goal:", sorted(p.goal))
print("applicable now:", p.actions(p.initial))

goal: ['on(a,b)', 'on(b,c)']
applicable now: [pick-up(a), pick-up(b), pick-up(c)]


## 4 · The delete-relaxation heuristic — derived, not written

**Ignore all delete effects.** Actions then only ever *add* facts: nothing can be
undone, no action can interfere with another, and the relaxed problem is solvable
in polynomial time.

> **Every admissible heuristic is the exact solution to an easier problem.**
> Week 4, returning — but here the *machine* performs the relaxation, from the
> domain description. **You write no heuristic at all.**

In [4]:
def h_add(state, goal, actions):
    """Sum over goal fluents of their relaxed reachability cost. NOT admissible
    (double-counts shared subgoals) but very informative."""
    cost = {f: 0 for f in state}
    changed = True
    while changed:
        changed = False
        for a in actions:
            if a.precond <= set(cost):
                c = 1 + sum(cost[p] for p in a.precond)
                for fl in a.add:                    # add effects only -- relaxed
                    if c < cost.get(fl, float("inf")):
                        cost[fl] = c
                        changed = True
    if not goal <= set(cost):
        return float("inf")                          # unreachable even relaxed
    return sum(cost[g] for g in goal)


def h_max(state, goal, actions):
    """Cost of the most expensive goal fluent. ADMISSIBLE."""
    cost = {f: 0 for f in state}
    changed = True
    while changed:
        changed = False
        for a in actions:
            if a.precond <= set(cost):
                c = 1 + max((cost[p] for p in a.precond), default=0)
                for fl in a.add:
                    if c < cost.get(fl, float("inf")):
                        cost[fl] = c
                        changed = True
    if not goal <= set(cost):
        return float("inf")
    return max((cost[g] for g in goal), default=0)


p = blocks_problem(3)
print("h_max(initial) =", h_max(p.initial, p.goal, p.action_list))
print("h_add(initial) =", h_add(p.initial, p.goal, p.action_list))
print("h_zero         = 0   (blind search)")

h_max(initial) = 2
h_add(initial) = 4
h_zero         = 0   (blind search)


## 5 · Week 4's A\*, unmodified

Read this cell and compare it to week 4. **It is the same function.** The only
difference is what gets passed in.

In [5]:
def astar(problem, h):
    """IDENTICAL to week 4's astar. Not a variant -- the same code."""
    start = (h(problem.initial), 0, problem.initial, None, None, 0)
    frontier = [start]
    best_g = {problem.initial: 0}
    parents = {}
    expansions, tiebreak = 0, 0

    while frontier:
        f, _, state, parent, action, g = heapq.heappop(frontier)
        if state in parents and g > best_g.get(state, float("inf")):
            continue
        parents[state] = (parent, action)
        if problem.is_goal(state):
            plan = []
            while action is not None:
                plan.append(action)
                parent, action = parents[parent]
            return plan[::-1], expansions
        expansions += 1
        for a in problem.actions(state):
            s2 = problem.result(state, a)
            g2 = g + problem.step_cost(state, a)
            if g2 < best_g.get(s2, float("inf")):
                best_g[s2] = g2
                tiebreak += 1
                heapq.heappush(frontier, (g2 + h(s2), tiebreak, s2, state, a, g2))
    return None, expansions


def plan(problem, heuristic="add"):
    hf = {"add": h_add, "max": h_max,
          "zero": lambda s, g, a: 0}[heuristic]
    return astar(problem, lambda s: hf(s, problem.goal, problem.action_list))


result, exp = plan(blocks_problem(3), "add")
print(f"plan ({len(result)} steps, {exp} expansions):")
for i, a in enumerate(result, 1):
    print(f"  {i}. {a}")

plan (4 steps, 6 expansions):
  1. pick-up(b)
  2. stack(b,c)
  3. pick-up(a)
  4. stack(a,b)


## 6 · The heuristic earns its keep

Same planner, three heuristics, growing problems.

In [6]:
print(f"  {'blocks':>7}{'heuristic':>12}{'plan len':>10}{'expansions':>12}{'time':>9}")
print("  " + "-" * 52)
for n in (3, 4, 5):
    for hname in ("zero", "max", "add"):
        p = blocks_problem(n)
        t0 = time.perf_counter()
        result, exp = plan(p, hname)
        dt = time.perf_counter() - t0
        n_steps = len(result) if result else -1
        print(f"  {n:>7}{hname:>12}{n_steps:>10}{exp:>12,}{dt:>8.3f}s")
    print()

print("""The heuristic was DERIVED FROM THE DOMAIN FILE. Nobody wrote a
blocks-world-specific heuristic -- h_add would work unchanged on logistics,
elevator scheduling, or Rubik's cube, given a domain description.

THAT is what a good representation buys: automatic heuristic derivation.
It is the strongest argument in this course for caring how you write a problem
down.""")

   blocks   heuristic  plan len  expansions     time
  ----------------------------------------------------
        3        zero         4          19   0.000s
        3         max         4           7   0.001s
        3         add         4           6   0.000s

        4        zero         6         118   0.000s
        4         max         6          26   0.005s
        4         add         6          12   0.002s

        5        zero         8         841   0.004s
        5         max         8         215   0.053s
        5         add         8          26   0.007s

The heuristic was DERIVED FROM THE DOMAIN FILE. Nobody wrote a
blocks-world-specific heuristic -- h_add would work unchanged on logistics,
elevator scheduling, or Rubik's cube, given a domain description.

THAT is what a good representation buys: automatic heuristic derivation.
It is the strongest argument in this course for caring how you write a problem
down.


## 7 · ⚠️ What a planner gives you that nothing else does

Ask for an **impossible** goal: `on(a,b)` **and** `on(b,a)`.

In [7]:
imp = PlanningProblem(
    {"ontable(a)", "ontable(b)", "clear(a)", "clear(b)", "handempty"},
    {"on(a,b)", "on(b,a)"},
    blocks_actions("ab"))

t0 = time.perf_counter()
result, exp = plan(imp, "add")
dt = time.perf_counter() - t0
print(f"result: {result}   after {exp} expansions in {dt:.3f}s")

print("""
The planner did not guess, hedge, or apologise. It searched the reachable state
space, exhausted it, and returned None.

  EXHAUSTION IS A PROOF. "No plan exists" is a THEOREM here.

There is no LLM analogue for this output. Ask a language model whether a goal is
achievable and it will produce a confident sentence either way; it cannot exhaust
a space. Put this on the scorecard as a CATEGORICAL difference, not a percentage.""")

result: None   after 5 expansions in 0.000s

The planner did not guess, hedge, or apologise. It searched the reachable state
space, exhausted it, and returned None.

  EXHAUSTION IS A PROOF. "No plan exists" is a THEOREM here.

There is no LLM analogue for this output. Ask a language model whether a goal is
achievable and it will produce a confident sentence either way; it cannot exhaust
a space. Put this on the scorecard as a CATEGORICAL difference, not a percentage.


## 8 · The sound verifier — 15 lines that carry the whole course

This function is the reason week 9 exists. **`verify()` never trusts the plan's
source.**

In [8]:
def verify(plan_steps, problem):
    """Sound plan validator. Returns (ok, first_error)."""
    state = problem.initial
    ground = {a.name: a for a in problem.action_list}
    for i, step in enumerate(plan_steps):
        name = step if isinstance(step, str) else step.name
        if name not in ground:
            return False, f"step {i}: unknown action {name!r}"
        action = ground[name]
        if not applicable(action, state):
            missing = sorted(action.precond - state)
            return False, f"step {i}: {name} needs {missing}"
        state = apply_action(action, state)
    if not problem.goal <= state:
        return False, f"goal not reached; missing {sorted(problem.goal - state)}"
    return True, None


p = blocks_problem(3)
good, _ = plan(p, "add")
print("valid plan     :", verify(good, p))

# Three ways a plan can be wrong -- all caught.
print("hallucinated   :", verify(["teleport(a,b)"], p))
print("bad precondition:", verify(["stack(a,b)"], p))          # not holding a
print("incomplete     :", verify(good[:-1], p))                # stops one short

print("""
Every failure is caught, LOCALISED, and explained. That error message is what
gets fed back to the LLM in arm D of the studio.""")

valid plan     : (True, None)
hallucinated   : (False, "step 0: unknown action 'teleport(a,b)'")
bad precondition: (False, "step 0: stack(a,b) needs ['holding(a)']")
incomplete     : (False, "goal not reached; missing ['on(a,b)']")

Every failure is caught, LOCALISED, and explained. That error message is what
gets fed back to the LLM in arm D of the studio.


## 9 · The studio, in miniature — arm D is the point

| Arm | Pipeline |
|---|---|
| A | hand-written PDDL → planner |
| B | LLM writes the **problem** file (domain given) → planner |
| C | LLM outputs a plan **directly** |
| **D** | **LLM proposes → `verify()` checks → error fed back → retry (max 3)** |

Simulated below with a deliberately unreliable "LLM" so you can watch the loop
work without burning tokens.

In [9]:
import random

def fake_llm_plan(problem, attempt, rng):
    """Stands in for a language model: plausible, sometimes wrong."""
    correct, _ = plan(problem, "add")
    if attempt == 0:
        return ["stack(a,b)"] + [s.name for s in correct]      # missing pick-up
    if attempt == 1:
        return [s.name for s in correct][:-1]                  # stops one short
    return [s.name for s in correct]                           # finally right


def arm_d(problem, max_retries=3, rng=None):
    rng = rng or random.Random(0)
    log = []
    for attempt in range(max_retries):
        proposal = fake_llm_plan(problem, attempt, rng)
        ok, err = verify(proposal, problem)
        log.append((attempt + 1, ok, err))
        if ok:
            return proposal, log
    return None, log


p = blocks_problem(3)
final, log = arm_d(p)
for attempt, ok, err in log:
    print(f"  attempt {attempt}: {'ACCEPTED' if ok else 'REJECTED -- ' + err}")
print(f"\nfinal plan valid: {verify(final, p)[0]}")

  attempt 1: REJECTED -- step 0: stack(a,b) needs ['holding(a)']
  attempt 2: REJECTED -- goal not reached; missing ['on(a,b)']
  attempt 3: ACCEPTED

final plan valid: True


## 10 · The question that makes this week land

> # In arm D, where does the guarantee come from?

## The verifier.

The LLM's output is a **proposal**. Correctness comes from a **sound checker** that
would have rejected a wrong plan no matter how confidently it was produced.

> ## An unreliable generator plus a sound verifier can be a reliable system.

**That is how you build reliable systems out of unreliable components, and it is
the single most transferable idea in this course.** It is also the answer to *"why
learn this if LLMs exist"* — **classical AI supplies the verifiers.**

---

## 11 · One more lesson, guaranteed to happen in the studio

A pair omits a precondition, and the planner returns a **valid** plan in which two
blocks are held in one hand.

> ## The planner is faithful to the model you wrote, not to the world you meant.

Name it out loud when it happens.

In [10]:
# Drop "handempty" from pick-up's preconditions and watch the planner
# cheerfully exploit the bug.
broken = [a if not a.name.startswith("pick-up")
          else Action(a.name, a.precond - {"handempty"}, a.add, a.delete)
          for a in blocks_actions("abc")]

bp = PlanningProblem(
    {"ontable(a)", "ontable(b)", "ontable(c)",
     "clear(a)", "clear(b)", "clear(c)", "handempty"},
    {"holding(a)", "holding(b)"},          # hold TWO blocks at once
    broken)

result, exp = plan(bp, "add")
print("plan to hold two blocks simultaneously:", result)
print("verify() says:", verify(result, bp))
print("""
Both the planner AND the verifier accept it -- because both are faithful to the
same flawed model. A sound verifier protects you from a bad GENERATOR, not from a
bad SPECIFICATION.

That distinction is worth a full minute. It is the limit of the week's thesis, and
students who understand it will not oversell verification in their capstone.""")

plan to hold two blocks simultaneously: [pick-up(a), pick-up(b)]
verify() says: (True, None)

Both the planner AND the verifier accept it -- because both are faithful to the
same flawed model. A sound verifier protects you from a bad GENERATOR, not from a
bad SPECIFICATION.

That distinction is worth a full minute. It is the limit of the week's thesis, and
students who understand it will not oversell verification in their capstone.
